# **Mengenal Tiga Jenis Klasifikasi: Binary, Multi-class, dan Multi-label**

Tugas Machine Learning (Klasifikasi).

Isi notebook:
1. Perbedaan **binary**, **multi-class**, dan **multi-label**
2. Sembilan dataset (3 untuk tiap jenis) beserta narasi dan **asal datasetnya**
3. Identifikasi jenis klasifikasi tiap dataset memakai **pandas**

## 1. Perbedaan Ketiga Jenis Klasifikasi

| Aspek | Binary | Multi-class | Multi-label |
|---|---|---|---|
| Banyak kelas/label yang mungkin | Tepat 2 | Lebih dari 2 | 2 atau lebih |
| Label per satu data | Tepat 1 | Tepat 1 | 0, 1, atau beberapa sekaligus |
| Hubungan antar kelas | Saling lepas | Saling lepas | Boleh muncul bersamaan |
| Cara berpikir | "Ya **atau** tidak" | "A **atau** B **atau** C" | "A **dan** B **dan** C" |
| Contoh | Uang kertas asli atau palsu | Angka tulisan tangan 0 sampai 9 | Satu bendera bisa berwarna merah **dan** putih **dan** biru |
| Bentuk target `y` | Satu kolom, 2 nilai unik | Satu kolom, lebih dari 2 nilai unik | Beberapa kolom 0/1 (matriks `n_data x n_label`) |
| Output model (jaringan saraf) | 1 neuron sigmoid | Softmax, ambil `argmax` | Satu sigmoid per label, lalu threshold |
| Loss yang lazim | Binary cross-entropy | Categorical cross-entropy | Binary cross-entropy per label |
| Metrik yang lazim | Accuracy, precision, recall, F1, ROC-AUC | Accuracy, macro/weighted F1, confusion matrix | Hamming loss, subset accuracy, F1 micro/macro |
| Cek dengan pandas | `nunique() == 2` | `nunique() > 2` | ada baris dengan `sum(axis=1) > 1` |

**Pembeda paling penting:** multi-class dan multi-label sama-sama punya banyak kelas, tetapi pada multi-class satu data hanya boleh punya **satu** kelas, sedangkan pada multi-label satu data boleh punya **lebih dari satu** label.

## 2. Persiapan

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)

GITHUB = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/"
HF = "https://huggingface.co/datasets/imodels/multitask-tabular-datasets/resolve/main/"


def deteksi_jenis(y):
    """Tentukan jenis klasifikasi dari DataFrame target `y` (hanya dengan pandas).

    - 1 kolom : 2 nilai unik -> binary, lebih dari 2 -> multi-class
    - >1 kolom (0/1): ada baris dengan lebih dari 1 label aktif -> multi-label
    """
    if y.shape[1] == 1:
        n = y.iloc[:, 0].nunique()
        if n == 2:
            return "binary"
        if n > 2:
            return "multi-class"
        return "tidak valid (hanya 1 kelas)"
    if not set(pd.unique(y.to_numpy().ravel())).issubset({0, 1}):
        return "bukan klasifikasi biasa"
    if (y.sum(axis=1) > 1).any():
        return "multi-label"
    return "multi-class (one-hot)"


# ---- loader: tiap fungsi mengembalikan (fitur, target) berupa DataFrame ----
def muat_github(nama_file, kolom, kolom_target, pemetaan=None):
    """CSV tanpa header dari repo GitHub, lalu diberi nama kolom."""
    df = pd.read_csv(GITHUB + nama_file, header=None, names=kolom)
    y = df[kolom_target]
    if pemetaan:
        y = y.map(pemetaan)
    return df.drop(columns=kolom_target), y.to_frame()


def muat_digits():
    from sklearn.datasets import load_digits
    d = load_digits(as_frame=True)
    return d.data, d.target.rename("angka").to_frame()


def muat_hf(nama):
    """Dataset multi-label dari salinan CSV di Hugging Face (kolom target berakhiran __target)."""
    df = pd.read_csv(f"{HF}{nama}.csv")
    kolom_y = [c for c in df.columns if str(c).endswith("__target")]
    if not kolom_y:
        raise ValueError("Kolom target (__target) tidak ditemukan")
    y = df[kolom_y].astype(int)
    y.columns = [str(c).replace("__target", "") for c in kolom_y]
    return df.drop(columns=kolom_y), y


RINGKASAN = []


def analisis(jenis_seharusnya, nama, sumber, loader):
    """Muat dataset, tampilkan identifikasi pandas, dan simpan ringkasannya."""
    print(f"=== {nama} ===")
    try:
        X, y = loader()
    except Exception as e:
        print(f"Gagal memuat: {type(e).__name__}: {e}\n")
        return

    jenis = deteksi_jenis(y)
    print(f"Fitur       : {X.shape[0]} baris x {X.shape[1]} kolom")
    print(f"Nilai kosong: {int(X.isnull().sum().sum())} sel pada fitur")

    if y.shape[1] == 1:
        s = y.iloc[:, 0]
        n_kelas = s.nunique()
        print(f"Target      : kolom '{y.columns[0]}' ({n_kelas} nilai unik)")
        print(f"Terdeteksi  : {jenis}")
        vc = s.value_counts()
        print(f"Rasio kelas terbesar : terkecil = {vc.max() / vc.min():.2f} : 1")
        print(vc.to_string())
    else:
        n_kelas = y.shape[1]
        per_baris = y.sum(axis=1)
        print(f"Target      : {n_kelas} kolom biner -> {', '.join(map(str, y.columns[:5]))}, ...")
        print(f"Terdeteksi  : {jenis}")
        print(f"Label aktif per data: rata-rata {per_baris.mean():.2f}, maks {per_baris.max()}")
        print("Sampel menurut jumlah label aktif:")
        print(per_baris.value_counts().sort_index().to_string())
        print("Data per label (terbanyak):")
        print(y.sum().sort_values(ascending=False).head(5).to_string())
    print("Contoh 5 baris target:")
    print(y.head().to_string())
    print()

    RINGKASAN.append({
        "Dataset": nama, "Seharusnya": jenis_seharusnya, "Terdeteksi": jenis,
        "Baris": X.shape[0], "Fitur": X.shape[1], "Kelas/Label": n_kelas, "Sumber": sumber,
    })

## 3. Klasifikasi Binary

Targetnya hanya punya **2 kelas** dan tiap data pasti jatuh ke salah satunya.

### 3.1 Pima Indians Diabetes

**Narasi:** Berisi 768 catatan medis perempuan keturunan suku Pima (usia minimal 21 tahun). Ada 8 fitur: jumlah kehamilan, kadar glukosa, tekanan darah, tebal lipatan kulit, kadar insulin, BMI, riwayat diabetes keluarga, dan usia. Target `outcome` bernilai 1 jika pasien didiagnosis diabetes dan 0 jika tidak (500 tidak, 268 ya). Hanya ada dua kemungkinan, sehingga jenisnya **binary**.

- **Kolom target:** `outcome` (0 = tidak diabetes, 1 = diabetes)
- **Asal dataset:** National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK), dipublikasikan luas lewat UCI/Kaggle.
- **Salinan yang dipakai:** https://github.com/jbrownlee/Datasets (`pima-indians-diabetes.csv`)

### 3.2 Banknote Authentication

**Narasi:** Berisi 1372 gambar uang kertas yang difoto, lalu tiap gambar diubah dengan *wavelet transform* menjadi 4 fitur numerik: variance, skewness, curtosis, dan entropy. Tujuannya membedakan uang kertas **asli** dan **palsu** (762 kelas 0 dan 610 kelas 1). Dua kelas saja, jadi **binary**.

- **Kolom target:** `kelas` (0 dan 1, yaitu asli dan palsu)
- **Asal dataset:** UCI Machine Learning Repository (id=267), disumbangkan oleh Volker Lohweg (University of Applied Sciences Ostwestfalen-Lippe).
- **Link:** https://archive.ics.uci.edu/dataset/267
- **Salinan yang dipakai:** https://github.com/jbrownlee/Datasets (`banknote_authentication.csv`)

### 3.3 Sonar (Mines vs. Rocks)

**Narasi:** Berisi 208 pantulan sinyal sonar dari dua jenis benda di dasar laut: **ranjau logam (M)** sebanyak 111 dan **batu (R)** sebanyak 97. Tiap sinyal dicirikan 60 angka, masing-masing energi pada satu pita frekuensi (nilai 0 sampai 1). Fiturnya banyak tetapi datanya sedikit, dan kelasnya cuma dua, jadi **binary**.

- **Kolom target:** `jenis` (M = mine/ranjau, R = rock/batu)
- **Asal dataset:** UCI Machine Learning Repository "Connectionist Bench (Sonar, Mines vs. Rocks)" (id=151). Pembuat: R. Paul Gorman dan Terrence Sejnowski, 1988.
- **Link:** https://archive.ics.uci.edu/dataset/151
- **Salinan yang dipakai:** https://github.com/jbrownlee/Datasets (`sonar.csv`)

In [2]:
analisis("binary", "Pima Indians Diabetes",
         "NIDDK (salinan GitHub jbrownlee)",
         lambda: muat_github(
             "pima-indians-diabetes.csv",
             ["hamil", "glukosa", "tekanan_darah", "tebal_kulit", "insulin",
              "bmi", "riwayat_keluarga", "usia", "outcome"],
             "outcome"))

analisis("binary", "Banknote Authentication",
         "UCI id=267 (salinan GitHub jbrownlee)",
         lambda: muat_github(
             "banknote_authentication.csv",
             ["variance", "skewness", "curtosis", "entropy", "kelas"],
             "kelas"))

analisis("binary", "Sonar (Mines vs. Rocks)",
         "UCI id=151 (salinan GitHub jbrownlee)",
         lambda: muat_github(
             "sonar.csv",
             [f"pita_{i}" for i in range(1, 61)] + ["jenis"],
             "jenis"))

=== Pima Indians Diabetes ===
Fitur       : 768 baris x 8 kolom
Nilai kosong: 0 sel pada fitur
Target      : kolom 'outcome' (2 nilai unik)
Terdeteksi  : binary
Rasio kelas terbesar : terkecil = 1.87 : 1
outcome
0    500
1    268
Contoh 5 baris target:
   outcome
0        1
1        0
2        1
3        0
4        1

=== Banknote Authentication ===
Fitur       : 1372 baris x 4 kolom
Nilai kosong: 0 sel pada fitur
Target      : kolom 'kelas' (2 nilai unik)
Terdeteksi  : binary
Rasio kelas terbesar : terkecil = 1.25 : 1
kelas
0    762
1    610
Contoh 5 baris target:
   kelas
0      0
1      0
2      0
3      0
4      0

=== Sonar (Mines vs. Rocks) ===
Fitur       : 208 baris x 60 kolom
Nilai kosong: 0 sel pada fitur
Target      : kolom 'jenis' (2 nilai unik)
Terdeteksi  : binary
Rasio kelas terbesar : terkecil = 1.14 : 1
jenis
M    111
R     97
Contoh 5 baris target:
  jenis
0     R
1     R
2     R
3     R
4     R



## 4. Klasifikasi Multi-class

Ada **lebih dari 2 kelas**, tetapi tiap data tetap hanya punya **satu** kelas.

### 4.1 Digits (Angka Tulisan Tangan)

**Narasi:** Berisi 1797 citra angka tulisan tangan berukuran 8 x 8 piksel. Tiap citra diratakan menjadi 64 fitur (intensitas piksel 0 sampai 16). Target adalah angka **0, 1, 2, ..., 9** (10 kelas) dengan jumlah per kelas hampir sama. Satu gambar hanya mewakili satu angka, jadi ini **multi-class**.

- **Kolom target:** `angka` (0 sampai 9)
- **Asal dataset:** UCI "Optical Recognition of Handwritten Digits" (id=80). Pembuat: E. Alpaydin dan C. Kaynak (Bogazici University), 1998.
- **Link:** https://archive.ics.uci.edu/dataset/80
- **Cara dimuat di sini:** `sklearn.datasets.load_digits` (salinan bagian uji dataset UCI tersebut).

### 4.2 Wheat Seeds

**Narasi:** Berisi 210 butir gandum dari 3 varietas, masing-masing 70 butir: **Kama, Rosa, dan Canadian**. Ukuran tiap butir diperoleh dari citra sinar-X lunak, lalu dihitung 7 fitur geometri: luas, keliling, kekompakan, panjang butir, lebar butir, koefisien asimetri, dan panjang alur butir. Satu butir hanya masuk satu varietas, dan kelasnya seimbang sempurna. Jenisnya **multi-class**.

- **Kolom target:** `varietas` (Kama / Rosa / Canadian)
- **Asal dataset:** UCI Machine Learning Repository "Seeds" (id=236). Pembuat: M. Charytanowicz dkk. (Lublin University of Technology), 2010.
- **Link:** https://archive.ics.uci.edu/dataset/236
- **Salinan yang dipakai:** https://github.com/jbrownlee/Datasets (`wheat-seeds.csv`)

### 4.3 Glass Identification

**Narasi:** Berisi 214 pecahan kaca dari TKP yang dianalisis secara kimia untuk keperluan forensik. Ada 9 fitur: indeks bias (RI) dan kadar oksida/unsur Na, Mg, Al, Si, K, Ca, Ba, Fe. Target adalah **6 tipe kaca** (jendela bangunan *float*, jendela bangunan *non-float*, jendela kendaraan *float*, wadah, peralatan makan, dan lampu depan). Sebarannya sangat timpang: tipe terbanyak 76 data, tipe tersedikit hanya 9. Satu pecahan hanya punya satu tipe, jadi ini **multi-class** yang tidak seimbang.

- **Kolom target:** `tipe_kaca`
- **Asal dataset:** UCI Machine Learning Repository "Glass Identification" (id=42). Pembuat: B. German (Central Research Establishment, Home Office Forensic Science Service, Inggris).
- **Link:** https://archive.ics.uci.edu/dataset/42
- **Salinan yang dipakai:** https://github.com/jbrownlee/Datasets (`glass.csv`)

In [3]:
analisis("multi-class", "Digits",
         "UCI id=80 (via scikit-learn)",
         muat_digits)

analisis("multi-class", "Wheat Seeds",
         "UCI id=236 (salinan GitHub jbrownlee)",
         lambda: muat_github(
             "wheat-seeds.csv",
             ["luas", "keliling", "kekompakan", "panjang_butir", "lebar_butir",
              "asimetri", "panjang_alur", "varietas"],
             "varietas",
             pemetaan={1: "Kama", 2: "Rosa", 3: "Canadian"}))

analisis("multi-class", "Glass Identification",
         "UCI id=42 (salinan GitHub jbrownlee)",
         lambda: muat_github(
             "glass.csv",
             ["RI", "Na", "Mg", "Al", "Si", "K", "Ca", "Ba", "Fe", "tipe_kaca"],
             "tipe_kaca",
             pemetaan={1: "jendela bangunan float", 2: "jendela bangunan non-float",
                       3: "jendela kendaraan float", 5: "wadah", 6: "peralatan makan",
                       7: "lampu depan"}))

=== Digits ===
Fitur       : 1797 baris x 64 kolom
Nilai kosong: 0 sel pada fitur
Target      : kolom 'angka' (10 nilai unik)
Terdeteksi  : multi-class
Rasio kelas terbesar : terkecil = 1.05 : 1
angka
3    183
1    182
5    182
4    181
6    181
9    180
7    179
0    178
2    177
8    174
Contoh 5 baris target:
   angka
0      0
1      1
2      2
3      3
4      4

=== Wheat Seeds ===
Fitur       : 210 baris x 7 kolom
Nilai kosong: 0 sel pada fitur
Target      : kolom 'varietas' (3 nilai unik)
Terdeteksi  : multi-class
Rasio kelas terbesar : terkecil = 1.00 : 1
varietas
Kama        70
Rosa        70
Canadian    70
Contoh 5 baris target:
  varietas
0     Kama
1     Kama
2     Kama
3     Kama
4     Kama

=== Glass Identification ===
Fitur       : 214 baris x 9 kolom
Nilai kosong: 0 sel pada fitur
Target      : kolom 'tipe_kaca' (6 nilai unik)
Terdeteksi  : multi-class
Rasio kelas terbesar : terkecil = 8.44 : 1
tipe_kaca
jendela bangunan non-float    76
jendela bangunan float        70
l

## 5. Klasifikasi Multi-label

Satu data boleh punya **banyak label sekaligus** (A **dan** B **dan** C). Label disimpan sebagai **kolom-kolom biner** (1 = aktif, 0 = tidak).

Ketiga dataset di bagian ini berasal dari *Multi-Label Classification Dataset Repository* (Universidad de Córdoba, https://www.uco.es/kdis/mllresources) dan dimuat dari salinan CSV di Hugging Face: https://huggingface.co/datasets/imodels/multitask-tabular-datasets (file `flags.csv`, `foodtruck.csv`, `water-quality.csv`). Kolom label di file tersebut berakhiran `__target`. Dataset ini butuh koneksi internet saat dijalankan.

### 5.1 Flags

**Narasi:** Berisi 194 negara beserta ciri-ciri bendera dan datanya. Ada 19 fitur, misalnya benua (`landmass`), zona, luas wilayah, populasi, bahasa, agama, jumlah garis, jumlah warna, dan ada tidaknya lingkaran, salib, bulan sabit, atau teks. Tugasnya memprediksi **warna apa saja yang muncul di bendera**, dengan 7 label: merah, hijau, biru, kuning, putih, hitam, dan oranye. Satu bendera hampir selalu memuat beberapa warna sekaligus (rata-rata sekitar 3,4 label per bendera), misalnya merah **dan** putih **dan** biru. Karena itu jenisnya **multi-label**.

- **Kolom label:** 7 kolom biner (warna bendera)
- **Asal dataset:** UCI Machine Learning Repository "Flags" (id=40), dipakai pertama kali untuk multi-label oleh Gonçalves dkk. (2013).
- **Link:** https://archive.ics.uci.edu/dataset/40

### 5.2 Foodtruck

**Narasi:** Dibuat dari jawaban kuesioner 407 responden di Brasil yang ditemui di festival makanan cepat saji dan acara populer, atau mengisi secara anonim. Kuesioner (dalam bahasa Portugis) menanyakan data pribadi dan preferensi mereka saat memilih *food truck*. Datanya punya 21 fitur dan 12 label, dengan rata-rata sekitar 2,3 label aktif per responden. Seorang responden bisa memilih beberapa hal sekaligus, jadi ini **multi-label**. Dataset ini termasuk jenis data rekomendasi.

- **Kolom label:** 12 kolom biner (preferensi responden)
- **Asal dataset:** Rivolli dkk. (2017), dikumpulkan di *Multi-Label Classification Dataset Repository*.
- **Link:** https://www.uco.es/kdis/mllresources

### 5.3 Water Quality

**Narasi:** Dataset ini dipakai untuk memprediksi kualitas air sungai di Slovenia dari 16 karakteristik fisik dan kimia, antara lain suhu, pH, kesadahan, NO2, dan CO2. Ada 1060 sampel dengan 14 label, dan rata-rata satu sampel punya sekitar 5,1 label aktif, jauh lebih banyak daripada Flags dan Foodtruck. Banyak label bisa aktif bersamaan pada satu sampel air, sehingga jenisnya **multi-label**.

- **Kolom label:** 14 kolom biner
- **Asal dataset:** Blockeel dkk. (1999), dikumpulkan di *Multi-Label Classification Dataset Repository*.
- **Link:** https://www.uco.es/kdis/mllresources

In [4]:
analisis("multi-label", "Flags",
         "UCI id=40 (via Multi-Label Repository, Hugging Face)",
         lambda: muat_hf("flags"))

analisis("multi-label", "Foodtruck",
         "Rivolli dkk. 2017 (via Multi-Label Repository, Hugging Face)",
         lambda: muat_hf("foodtruck"))

analisis("multi-label", "Water Quality",
         "Blockeel dkk. 1999 (via Multi-Label Repository, Hugging Face)",
         lambda: muat_hf("water-quality"))

=== Flags ===
Fitur       : 194 baris x 19 kolom
Nilai kosong: 0 sel pada fitur
Target      : 7 kolom biner -> red, green, blue, yellow, white, ...
Terdeteksi  : multi-label
Label aktif per data: rata-rata 3.39, maks 7
Sampel menurut jumlah label aktif:
1     1
2    43
3    76
4    44
5    15
6    13
7     2
Data per label (terbanyak):
red       153
white     146
blue       99
green      91
yellow     91
Contoh 5 baris target:
   red  green  blue  yellow  white  black  orange
0    1      0     0       1      0      1       0
1    1      0     1       1      1      0       1
2    1      0     0       1      0      1       0
3    0      0     1       0      1      0       0
4    0      0     1       1      0      1       0

=== Foodtruck ===
Fitur       : 407 baris x 21 kolom
Nilai kosong: 0 sel pada fitur
Target      : 12 kolom biner -> street_food, gourmet, italian_food, brazilian_food, mexican_food, ...
Terdeteksi  : multi-label
Label aktif per data: rata-rata 2.29, maks 11
Sampel men

## 6. Ringkasan Hasil Identifikasi

Tabel ini dibuat otomatis dari hasil cell-cell di atas. Kolom **Terdeteksi** seharusnya sama dengan kolom **Seharusnya**.

In [5]:
tabel = pd.DataFrame(RINGKASAN)
tabel["Cocok?"] = np.where(tabel["Seharusnya"] == tabel["Terdeteksi"], "Ya", "Tidak")
tabel

,Dataset,Seharusnya,Terdeteksi,Baris,Fitur,Kelas/Label,Sumber,Cocok?
0,Pima Indians Diabetes,binary,binary,768,8,2,NIDDK (salinan GitHub jbrownlee),Ya
1,Banknote Authentication,binary,binary,1372,4,2,UCI id=267 (salinan GitHub jbrownlee),Ya
2,Sonar (Mines vs. Rocks),binary,binary,208,60,2,UCI id=151 (salinan GitHub jbrownlee),Ya
3,Digits,multi-class,multi-class,1797,64,10,UCI id=80 (via scikit-learn),Ya
4,Wheat Seeds,multi-class,multi-class,210,7,3,UCI id=236 (salinan GitHub jbrownlee),Ya
5,Glass Identification,multi-class,multi-class,214,9,6,UCI id=42 (salinan GitHub jbrownlee),Ya
6,Flags,multi-label,multi-label,194,19,7,"UCI id=40 (via Multi-Label Repository, Hugging...",Ya
7,Foodtruck,multi-label,multi-label,407,21,12,"Rivolli dkk. 2017 (via Multi-Label Repository,...",Ya
8,Water Quality,multi-label,multi-label,1060,16,14,Blockeel dkk. 1999 (via Multi-Label Repository...,Ya


## 7. Kesimpulan

- **Binary:** target satu kolom dengan **2 nilai unik** (`nunique() == 2`). Contoh di sini: Pima Diabetes, Banknote, Sonar.
- **Multi-class:** target satu kolom dengan **lebih dari 2 nilai unik**, tiap data hanya satu kelas. Contoh: Digits, Wheat Seeds, Glass.
- **Multi-label:** target berupa **beberapa kolom biner** dan ada data yang punya **lebih dari satu label aktif** (`sum(axis=1).max() > 1`). Contoh: Flags, Foodtruck, Water Quality.
- Kelas yang tidak seimbang (misalnya Glass, rasio kelas terbesar dan terkecil sekitar 8 : 1) tidak mengubah jenis klasifikasinya, tetapi perlu diperhatikan saat memilih metrik.